In [36]:
import pandas as pd
import numpy as np

In [37]:
df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/flight_information/flights_info_gold.parquet")

In [38]:
df_traj = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet")

In [39]:
pd.set_option("display.max_columns", None)

In [40]:
df.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'arr_revised_time_utc', 'status', 'dep_lat',
       'dep_lon', 'dep_altitude', 'distance', 'haul', 'wake_turbulence_cat'],
      dtype='str')

In [41]:
df_traj.columns

Index(['flight_id', 'episode_id', 'point_timestamp', 'icao', 'date', 'e_m',
       'n_m', 'u_m', 'ux', 'uy', 'uz', 'r', 'sin_theta', 'cos_theta',
       'delta_t', 'distance_km', 'on_ground', 'segment_id', 'gap_flag'],
      dtype='str')

In [42]:
len(df)

83727

In [43]:
df_flights = df.copy()
df_traj = df_traj.copy()

In [44]:
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────
FLIGHTS_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/flight_information/flights_info_gold.parquet"
TRAJ_PATH    = "/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet"

OUTPUT_ALL_PATH   = "/Users/YGT/ist-airport-decision-support-system/data/gold/ist_flights_enriched_v6.parquet"
OUTPUT_LABEL_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/ist_flights_labeled_v6.parquet"

# ─────────────────────────────────────────────────────────────
# PARAMETRELER
# ─────────────────────────────────────────────────────────────
ENTRY_LOOKBACK_HRS          = 3
GAP_THRESHOLD_S             = 1800
LOCAL_MIN_RADIUS            = 5

POST_TERM_MIN_MIN           = 0
POST_TERM_MAX_MIN           = 120

ENTRY_DIST_MIN_KM           = 25
ENTRY_DIST_MAX_KM           = 125

LANDING_DIST_MAX_KM         = 18
TIME_DIFF_MAX_MIN           = 60

DIST_WEIGHT                 = 1.0
TIME_WEIGHT                 = 0.15

# Arrival-like trend
PRE_TREND_MINUTES           = 10
POST_TREND_MINUTES          = 10
MIN_TREND_POINTS            = 5
PRE_DECREASE_RATIO_MIN      = 0.60
POST_DECREASE_RATIO_MAX     = 0.40

# Holding pattern tespiti
HOLDING_WINDOW_MIN          = 30    # landing öncesi kaç dk'ya bakılacak
HOLDING_STD_MIN_KM          = 5.0   # mesafe std bu değerin üzerindeyse holding var
HOLDING_N_REVERSALS_MIN     = 2     # en az kaç yön değişimi olmalı (ek kontrol)

LANDING_SEARCH_STAGES = [
    {"name": "stage_1_pm90m",  "window_min": 90,  "confidence": "high"},
    {"name": "stage_2_pm180m", "window_min": 180, "confidence": "medium"},
    {"name": "stage_3_pm300m", "window_min": 300, "confidence": "low"},
]

# ─────────────────────────────────────────────────────────────
# YARDIMCI FONKSİYONLAR
# ─────────────────────────────────────────────────────────────
def get_local_min_candidates(win: pd.DataFrame, local_min_radius: int) -> pd.DataFrame:
    win = win.sort_values("point_timestamp").reset_index(drop=True).copy()
    dist_vals = win["distance_km"].to_numpy()
    is_local_min = np.zeros(len(win), dtype=bool)

    for i in range(len(win)):
        left  = max(0, i - local_min_radius)
        right = min(len(win), i + local_min_radius + 1)
        neighborhood = dist_vals[left:right]
        if np.isfinite(dist_vals[i]) and dist_vals[i] == np.nanmin(neighborhood):
            is_local_min[i] = True

    cand = win[is_local_min].copy()
    if cand.empty:
        cand = win.copy()
    return cand


def score_landing_candidates(cand: pd.DataFrame, ref_time: pd.Timestamp) -> pd.DataFrame:
    cand = cand.copy()
    cand["time_diff_min"] = (
        (cand["point_timestamp"] - ref_time).dt.total_seconds() / 60.0
    )
    cand["time_diff_min_abs"] = cand["time_diff_min"].abs()
    cand["landing_score"] = (
        DIST_WEIGHT * cand["distance_km"] +
        TIME_WEIGHT * cand["time_diff_min_abs"]
    )
    return cand


def compute_decrease_ratio(series: pd.Series) -> float:
    diffs = series.diff().dropna()
    if len(diffs) == 0:
        return np.nan
    return float((diffs < 0).mean())


def detect_holding(grp: pd.DataFrame, landing_t: pd.Timestamp) -> dict:
    """
    Landing öncesi HOLDING_WINDOW_MIN dakikaya bakarak holding pattern tespiti.

    İki kriter:
      1) Mesafenin std > HOLDING_STD_MIN_KM  (dalgalanma var)
      2) Yön değişimi sayısı >= HOLDING_N_REVERSALS_MIN  (mesafe inip çıkıyor)

    İkisi de sağlanıyorsa holding=True.
    """
    t_lo = landing_t - pd.Timedelta(minutes=HOLDING_WINDOW_MIN)
    win = grp[
        (grp["point_timestamp"] >= t_lo) &
        (grp["point_timestamp"] <= landing_t)
    ].sort_values("point_timestamp").reset_index(drop=True)

    if len(win) < 4:
        return {
            "holding_detected": False,
            "holding_dist_std": np.nan,
            "holding_n_reversals": 0,
        }

    dist = win["distance_km"].to_numpy()
    dist_std = float(np.nanstd(dist))

    # Yön değişimi: diff işaret değişimi sayısı
    diffs = np.diff(dist)
    signs = np.sign(diffs[np.isfinite(diffs)])
    reversals = int(np.sum(np.diff(signs) != 0)) if len(signs) > 1 else 0

    holding = (
        dist_std >= HOLDING_STD_MIN_KM and
        reversals >= HOLDING_N_REVERSALS_MIN
    )

    return {
        "holding_detected": bool(holding),
        "holding_dist_std": dist_std,
        "holding_n_reversals": reversals,
    }


def evaluate_arrival_like(grp: pd.DataFrame, landing_t: pd.Timestamp) -> dict:
    """
    Landing etrafında arrival-like trend kontrolü.
    Holding varsa pre_decrease_ratio kontrolünü gevşet.
    """
    grp = grp.sort_values("point_timestamp").reset_index(drop=True)

    pre = grp[
        (grp["point_timestamp"] >= landing_t - pd.Timedelta(minutes=PRE_TREND_MINUTES)) &
        (grp["point_timestamp"] <= landing_t)
    ].sort_values("point_timestamp").reset_index(drop=True)

    post = grp[
        (grp["point_timestamp"] >= landing_t) &
        (grp["point_timestamp"] <= landing_t + pd.Timedelta(minutes=POST_TREND_MINUTES))
    ].sort_values("point_timestamp").reset_index(drop=True)

    holding_info = detect_holding(grp, landing_t)
    holding = holding_info["holding_detected"]

    pre_ratio  = np.nan
    post_ratio = np.nan
    arrival_like_pre  = False
    arrival_like_post = False

    if len(pre) >= MIN_TREND_POINTS:
        pre_ratio = compute_decrease_ratio(pre["distance_km"])
        if holding:
            # Holding varsa daha gevşek eşik — son yaklaşma kısa olabilir
            arrival_like_pre = pd.notna(pre_ratio) and (pre_ratio >= 0.45)
        else:
            arrival_like_pre = pd.notna(pre_ratio) and (pre_ratio >= PRE_DECREASE_RATIO_MIN)

    if len(post) >= MIN_TREND_POINTS:
        post_ratio = compute_decrease_ratio(post["distance_km"])
        arrival_like_post = pd.notna(post_ratio) and (post_ratio <= POST_DECREASE_RATIO_MAX)
    else:
        arrival_like_post = True  # post pencere kısa, trajectory kesik — geç

    arrival_like = bool(arrival_like_pre and arrival_like_post)

    return {
        "arrival_like_pre":              bool(arrival_like_pre),
        "arrival_like_post":             bool(arrival_like_post),
        "arrival_like":                  arrival_like,
        "arrival_pre_decrease_ratio":    pre_ratio,
        "arrival_post_decrease_ratio":   post_ratio,
        "pre_points":                    int(len(pre)),
        "post_points":                   int(len(post)),
        **holding_info,
    }


def find_landing_proxy_adaptive(grp: pd.DataFrame, ref_time: pd.Timestamp) -> dict | None:
    grp = grp.sort_values("point_timestamp").reset_index(drop=True)

    fallback_best         = None
    fallback_stage        = None
    fallback_cand_len     = np.nan
    fallback_arrival_eval = None

    for stage in LANDING_SEARCH_STAGES:
        window_min = stage["window_min"]
        t_lo = ref_time - pd.Timedelta(minutes=window_min)
        t_hi = ref_time + pd.Timedelta(minutes=window_min)

        win = grp[
            (grp["point_timestamp"] >= t_lo) &
            (grp["point_timestamp"] <= t_hi)
        ].copy()

        if win.empty:
            continue

        cand = get_local_min_candidates(win, LOCAL_MIN_RADIUS)
        cand = score_landing_candidates(cand, ref_time)
        cand = cand.sort_values("landing_score").reset_index(drop=True)

        for _, best in cand.iterrows():
            arrival_eval = evaluate_arrival_like(grp, best["point_timestamp"])

            if fallback_best is None:
                fallback_best         = best
                fallback_stage        = stage
                fallback_cand_len     = int(len(cand))
                fallback_arrival_eval = arrival_eval

            is_acceptable = (
                float(best["distance_km"]) <= LANDING_DIST_MAX_KM and
                abs(float(best["time_diff_min"])) <= TIME_DIFF_MAX_MIN and
                arrival_eval["arrival_like"]
            )

            if is_acceptable:
                return {
                    "landing_t":              best["point_timestamp"],
                    "landing_d":              float(best["distance_km"]),
                    "time_diff_min":          float(best["time_diff_min"]),
                    "landing_score":          float(best["landing_score"]),
                    "landing_search_stage":   stage["name"],
                    "match_confidence":       stage["confidence"],
                    "landing_candidate_count": int(len(cand)),
                    "landing_window_min":     int(window_min),
                    "landing_source":         "distance_time_local_min_arrival_like",
                    "landing_is_fallback":    False,
                    **arrival_eval,
                }

    # Hiç acceptable yoksa fallback
    if fallback_best is not None:
        return {
            "landing_t":              fallback_best["point_timestamp"],
            "landing_d":              float(fallback_best["distance_km"]),
            "time_diff_min":          float(fallback_best["time_diff_min"]),
            "landing_score":          float(fallback_best["landing_score"]),
            "landing_search_stage":   fallback_stage["name"],
            "match_confidence":       f'{fallback_stage["confidence"]}_fallback',
            "landing_candidate_count": fallback_cand_len,
            "landing_window_min":     int(fallback_stage["window_min"]),
            "landing_source":         "distance_time_local_min_fallback",
            "landing_is_fallback":    True,
            **fallback_arrival_eval,
        }

    return None


def extract_entry_segment(grp: pd.DataFrame, landing_t: pd.Timestamp):
    e_lo = landing_t - pd.Timedelta(hours=ENTRY_LOOKBACK_HRS)

    win = grp[
        (grp["point_timestamp"] >= e_lo) &
        (grp["point_timestamp"] <= landing_t)
    ].copy()

    if len(win) < 2:
        return None

    win = win.sort_values("point_timestamp").reset_index(drop=True)
    win["_gap_s"] = win["point_timestamp"].diff().dt.total_seconds()
    gap_pos = np.where(win["_gap_s"].fillna(0).to_numpy() > GAP_THRESHOLD_S)[0]

    if len(gap_pos) > 0:
        segment = win.iloc[gap_pos[-1]:].reset_index(drop=True).copy()
        segment_source = "after_last_big_gap"
    else:
        segment = win.copy()
        segment_source = "full_lookback_window_no_gap"

    return segment, segment_source


def choose_entry_proxy(segment: pd.DataFrame) -> dict | None:
    seg = segment.sort_values("point_timestamp").reset_index(drop=True).copy()
    if seg.empty:
        return None

    in_band = (
        (seg["distance_km"] >= ENTRY_DIST_MIN_KM) &
        (seg["distance_km"] <= ENTRY_DIST_MAX_KM)
    )
    if in_band.any():
        row = seg.iloc[np.where(in_band.to_numpy())[0][0]]
        return {"entry_t": row["point_timestamp"], "entry_d": float(row["distance_km"]),
                "entry_source": "first_point_inside_entry_band"}

    in_upper = seg["distance_km"] <= ENTRY_DIST_MAX_KM
    if in_upper.any():
        row = seg.iloc[np.where(in_upper.to_numpy())[0][0]]
        return {"entry_t": row["point_timestamp"], "entry_d": float(row["distance_km"]),
                "entry_source": "first_point_inside_tma_upper_only"}

    row = seg.iloc[0]
    return {"entry_t": row["point_timestamp"], "entry_d": float(row["distance_km"]),
            "entry_source": "segment_start_fallback"}


# ─────────────────────────────────────────────────────────────
# 1. VERİ YÜKLE
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("1. VERİ YÜKLEME")
print("=" * 60)

df_flights = pd.read_parquet(FLIGHTS_PATH)
df_traj    = pd.read_parquet(TRAJ_PATH)
print(f"  Flights : {len(df_flights):>10,}")
print(f"  Traj    : {len(df_traj):>10,}")

# ─────────────────────────────────────────────────────────────
# 2. FORMAT / TEMİZLİK
# ─────────────────────────────────────────────────────────────
for c in ["hex_icao", "arr_sched_time_utc"]:
    if c not in df_flights.columns:
        raise ValueError(f"Zorunlu kolon yok: {c}")

if "arr_revised_time_utc" not in df_flights.columns:
    df_flights["arr_revised_time_utc"] = pd.NaT

for c in ["arr_sched_time_utc", "arr_revised_time_utc"]:
    df_flights[c] = pd.to_datetime(df_flights[c], utc=True, errors="coerce")

df_traj["point_timestamp"] = pd.to_datetime(df_traj["point_timestamp"], utc=True, errors="coerce")
df_traj["distance_km"]     = pd.to_numeric(df_traj["distance_km"], errors="coerce")
df_flights["hex_icao"]     = df_flights["hex_icao"].astype(str).str.upper().str.strip()
df_traj["icao"]            = df_traj["icao"].astype(str).str.upper().str.strip()

df_flights["ref_time"] = df_flights["arr_revised_time_utc"].fillna(df_flights["arr_sched_time_utc"])
df_flights = df_flights.dropna(subset=["hex_icao", "arr_sched_time_utc", "ref_time"]).copy()
df_traj    = df_traj.dropna(subset=["icao", "point_timestamp", "distance_km"]).copy()

print(f"\n  Flights (temiz) : {len(df_flights):>10,}")
print(f"  Traj    (temiz) : {len(df_traj):>10,}")

# ─────────────────────────────────────────────────────────────
# 3. TRAJ GRUPLAMA
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. TRAJ GRUPLAMA")
print("=" * 60)

df_traj = df_traj.sort_values(["icao", "point_timestamp"], kind="mergesort")
traj_grouped = {
    icao: grp.reset_index(drop=True)
    for icao, grp in df_traj.groupby("icao", sort=False)
}
print(f"  Benzersiz ICAO : {len(traj_grouped):,}")

# ─────────────────────────────────────────────────────────────
# 4. LABEL EXTRACTION
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. LABEL EXTRACTION")
print("=" * 60)

_NAT = pd.NaT
_NAN = np.nan
results = []

for i, flight in enumerate(df_flights.itertuples(index=False), start=1):
    if i % 10_000 == 0:
        print(f"  [{i:,} / {len(df_flights):,}]")

    icao     = flight.hex_icao
    ref_time = flight.ref_time

    row = dict(
        actual_landing_time           = _NAT,
        landing_dist_km               = _NAN,
        actual_entry_time             = _NAT,
        entry_dist_km                 = _NAN,
        post_terminal_duration_s      = _NAN,
        post_terminal_duration_min    = _NAN,
        landing_proxy_time_diff_min   = _NAN,
        landing_score                 = _NAN,
        landing_candidate_count       = _NAN,
        landing_window_min            = _NAN,
        landing_source                = None,
        landing_search_stage          = None,
        match_confidence              = None,
        landing_is_fallback           = False,
        arrival_like_pre              = False,
        arrival_like_post             = False,
        arrival_like                  = False,
        arrival_pre_decrease_ratio    = _NAN,
        arrival_post_decrease_ratio   = _NAN,
        pre_points                    = _NAN,
        post_points                   = _NAN,
        holding_detected              = False,
        holding_dist_std              = _NAN,
        holding_n_reversals           = 0,
        entry_source                  = None,
        entry_segment_source          = None,
        is_matched                    = False,
        is_plausible_label            = False,
        match_quality_flag            = "no_match",
    )

    if icao not in traj_grouped:
        results.append(row)
        continue

    grp = traj_grouped[icao]

    # ── A) LANDING PROXY
    landing_info = find_landing_proxy_adaptive(grp, ref_time)
    if landing_info is None:
        results.append(row)
        continue

    landing_t = landing_info["landing_t"]
    landing_d = landing_info["landing_d"]
    time_diff = landing_info["time_diff_min"]

    row.update({
        "actual_landing_time":         landing_t,
        "landing_dist_km":             landing_d,
        "landing_proxy_time_diff_min": time_diff,
        "landing_score":               landing_info["landing_score"],
        "landing_candidate_count":     landing_info["landing_candidate_count"],
        "landing_window_min":          landing_info["landing_window_min"],
        "landing_source":              landing_info["landing_source"],
        "landing_search_stage":        landing_info["landing_search_stage"],
        "match_confidence":            landing_info["match_confidence"],
        "landing_is_fallback":         landing_info["landing_is_fallback"],
        "arrival_like_pre":            landing_info["arrival_like_pre"],
        "arrival_like_post":           landing_info["arrival_like_post"],
        "arrival_like":                landing_info["arrival_like"],
        "arrival_pre_decrease_ratio":  landing_info["arrival_pre_decrease_ratio"],
        "arrival_post_decrease_ratio": landing_info["arrival_post_decrease_ratio"],
        "pre_points":                  landing_info["pre_points"],
        "post_points":                 landing_info["post_points"],
        "holding_detected":            landing_info["holding_detected"],
        "holding_dist_std":            landing_info["holding_dist_std"],
        "holding_n_reversals":         landing_info["holding_n_reversals"],
        "is_matched":                  True,
    })

    # ── B) ENTRY SEGMENT
    seg_info = extract_entry_segment(grp, landing_t)
    if seg_info is None:
        row["match_quality_flag"] = "landing_found_but_no_entry_segment"
        results.append(row)
        continue

    segment, segment_source = seg_info
    row["entry_segment_source"] = segment_source

    # ── C) ENTRY PROXY
    entry_info = choose_entry_proxy(segment)
    if entry_info is None:
        row["match_quality_flag"] = "landing_found_but_no_entry"
        results.append(row)
        continue

    entry_t = entry_info["entry_t"]
    entry_d = entry_info["entry_d"]

    row["actual_entry_time"] = entry_t
    row["entry_dist_km"]     = entry_d
    row["entry_source"]      = entry_info["entry_source"]

    # ── D) TARGET
    post_s   = (landing_t - entry_t).total_seconds()
    post_min = post_s / 60.0

    row["post_terminal_duration_s"]   = post_s
    row["post_terminal_duration_min"] = post_min

    # ── E) PLAUSIBILITY
    ok = (
        POST_TERM_MIN_MIN <= post_min <= POST_TERM_MAX_MIN and
        ENTRY_DIST_MIN_KM <= entry_d  <= ENTRY_DIST_MAX_KM and
        landing_d         <= LANDING_DIST_MAX_KM            and
        abs(time_diff)    <= TIME_DIFF_MAX_MIN               and
        entry_t           <= landing_t                       and
        bool(row["arrival_like"])
    )

    row["is_plausible_label"] = ok

    if ok and row["match_confidence"] == "high":
        row["match_quality_flag"] = "good_match_high_conf"
    elif ok and row["match_confidence"] == "medium":
        row["match_quality_flag"] = "good_match_medium_conf"
    elif ok and row["match_confidence"] == "low":
        row["match_quality_flag"] = "good_match_low_conf"
    elif ok and str(row["match_confidence"]).endswith("_fallback"):
        row["match_quality_flag"] = "good_match_fallback"
    elif not bool(row["arrival_like"]):
        row["match_quality_flag"] = "bad_arrival_like_trend"
    elif landing_d > LANDING_DIST_MAX_KM:
        row["match_quality_flag"] = "bad_landing_distance"
    elif abs(time_diff) > TIME_DIFF_MAX_MIN:
        row["match_quality_flag"] = "bad_time_alignment"
    elif not (ENTRY_DIST_MIN_KM <= entry_d <= ENTRY_DIST_MAX_KM):
        row["match_quality_flag"] = "bad_entry_distance"
    elif not (POST_TERM_MIN_MIN <= post_min <= POST_TERM_MAX_MIN):
        row["match_quality_flag"] = "bad_post_terminal_duration"
    elif entry_t > landing_t:
        row["match_quality_flag"] = "entry_after_landing"
    else:
        row["match_quality_flag"] = "other"

    results.append(row)

# ─────────────────────────────────────────────────────────────
# 5. BİRLEŞTİR & KAYDET
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. BİRLEŞTİR & KAYDET")
print("=" * 60)

df_res   = pd.DataFrame(results)
df_final = pd.concat(
    [df_flights.drop(columns=["ref_time"]).reset_index(drop=True),
     df_res.reset_index(drop=True)],
    axis=1,
)
df_labeled = df_final[df_final["is_plausible_label"]].copy()

df_final.to_parquet(OUTPUT_ALL_PATH, index=False)
df_labeled.to_parquet(OUTPUT_LABEL_PATH, index=False)
print(f"  Tüm    → {OUTPUT_ALL_PATH}")
print(f"  Labeled → {OUTPUT_LABEL_PATH}")

# ─────────────────────────────────────────────────────────────
# 6. RAPOR
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("5. RAPOR")
print("=" * 60)

total         = len(df_final)
landing_found = int(df_final["actual_landing_time"].notna().sum())
entry_found   = int(df_final["actual_entry_time"].notna().sum())
matched       = int(df_final["is_matched"].sum())
labeled       = len(df_labeled)

print(f"  Toplam flight       : {total:>10,}")
print(f"  Landing bulunan     : {landing_found:>10,}  ({landing_found/total*100:.1f}%)")
print(f"  Entry bulunan       : {entry_found:>10,}  ({entry_found/total*100:.1f}%)")
print(f"  Matched             : {matched:>10,}  ({matched/total*100:.1f}%)")
print(f"  Plausible label     : {labeled:>10,}  ({labeled/total*100:.1f}%)")

print("\n  Match quality:")
print(df_final["match_quality_flag"].value_counts(dropna=False).to_string())

print("\n  Landing search stage:")
print(df_final["landing_search_stage"].value_counts(dropna=False).to_string())

print("\n  Match confidence:")
print(df_final["match_confidence"].value_counts(dropna=False).to_string())

print("\n  Arrival-like:")
print(df_final["arrival_like"].value_counts(dropna=False).to_string())

print("\n  Holding detected:")
print(df_final["holding_detected"].value_counts(dropna=False).to_string())

pct = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

if labeled > 0:
    print("\n  TARGET — post_terminal_duration_min:")
    print(df_labeled["post_terminal_duration_min"].describe(percentiles=pct).round(2).to_string())

    print("\n  entry_dist_km:")
    print(df_labeled["entry_dist_km"].describe(percentiles=pct).round(2).to_string())

    print("\n  landing_dist_km:")
    print(df_labeled["landing_dist_km"].describe(percentiles=pct).round(2).to_string())

    print("\n  landing_proxy_time_diff_min:")
    print(df_labeled["landing_proxy_time_diff_min"].describe(percentiles=pct).round(2).to_string())

    print("\n  holding_detected (labeled):")
    print(df_labeled["holding_detected"].value_counts(dropna=False).to_string())

    print("\n  holding_dist_std (labeled):")
    print(df_labeled["holding_dist_std"].describe(percentiles=pct).round(2).to_string())

    print("\n  plausible labels by search stage:")
    print(df_labeled["landing_search_stage"].value_counts(dropna=False).to_string())

    print("\n  entry_source:")
    print(df_labeled["entry_source"].value_counts(dropna=False).to_string())

    print("\n  entry_segment_source:")
    print(df_labeled["entry_segment_source"].value_counts(dropna=False).to_string())
else:
    print("\n  Uyarı: plausible label çıkmadı.")

1. VERİ YÜKLEME
  Flights :     83,727
  Traj    : 28,425,192

  Flights (temiz) :     83,727
  Traj    (temiz) : 28,425,192

2. TRAJ GRUPLAMA
  Benzersiz ICAO : 10,009

3. LABEL EXTRACTION
  [10,000 / 83,727]
  [20,000 / 83,727]
  [30,000 / 83,727]
  [40,000 / 83,727]
  [50,000 / 83,727]
  [60,000 / 83,727]
  [70,000 / 83,727]
  [80,000 / 83,727]

4. BİRLEŞTİR & KAYDET
  Tüm    → /Users/YGT/ist-airport-decision-support-system/data/gold/ist_flights_enriched_v6.parquet
  Labeled → /Users/YGT/ist-airport-decision-support-system/data/gold/ist_flights_labeled_v6.parquet

5. RAPOR
  Toplam flight       :     83,727
  Landing bulunan     :     67,164  (80.2%)
  Entry bulunan       :     58,068  (69.4%)
  Matched             :     67,164  (80.2%)
  Plausible label     :     30,883  (36.9%)

  Match quality:
match_quality_flag
good_match_high_conf                  30883
bad_landing_distance                  17016
no_match                              16563
landing_found_but_no_entry_segment   

In [45]:
print("Holding TRUE — post_terminal_duration_min:")
print(df_labeled[df_labeled["holding_detected"]]["post_terminal_duration_min"].describe(percentiles=[0.25,0.5,0.75]).round(2))

print("\nHolding FALSE — post_terminal_duration_min:")
print(df_labeled[~df_labeled["holding_detected"]]["post_terminal_duration_min"].describe(percentiles=[0.25,0.5,0.75]).round(2))

Holding TRUE — post_terminal_duration_min:
count    12318.00
mean        16.41
std          6.12
min          2.97
25%         11.98
50%         15.42
75%         20.00
max         57.15
Name: post_terminal_duration_min, dtype: float64

Holding FALSE — post_terminal_duration_min:
count    18565.00
mean        10.61
std          6.01
min          1.18
25%          6.35
50%          9.78
75%         13.25
max         56.73
Name: post_terminal_duration_min, dtype: float64


In [46]:
df_final.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'arr_revised_time_utc', 'status', 'dep_lat',
       'dep_lon', 'dep_altitude', 'distance', 'haul', 'wake_turbulence_cat',
       'actual_landing_time', 'landing_dist_km', 'actual_entry_time',
       'entry_dist_km', 'post_terminal_duration_s',
       'post_terminal_duration_min', 'landing_proxy_time_diff_min',
       'landing_score', 'landing_candidate_count', 'landing_window_min',
       'landing_source', 'landing_search_stage', 'match_confidence',
       'landing_is_fallback', 'arrival_like_pre', 'arrival_like_post',
       'arrival_like', 'arrival_pre_decrease_ratio',
       'arrival_p

In [47]:
df_labeled.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'arr_revised_time_utc', 'status', 'dep_lat',
       'dep_lon', 'dep_altitude', 'distance', 'haul', 'wake_turbulence_cat',
       'actual_landing_time', 'landing_dist_km', 'actual_entry_time',
       'entry_dist_km', 'post_terminal_duration_s',
       'post_terminal_duration_min', 'landing_proxy_time_diff_min',
       'landing_score', 'landing_candidate_count', 'landing_window_min',
       'landing_source', 'landing_search_stage', 'match_confidence',
       'landing_is_fallback', 'arrival_like_pre', 'arrival_like_post',
       'arrival_like', 'arrival_pre_decrease_ratio',
       'arrival_p

In [48]:
df_labeled.head()

,date,day_of_week,hour_ist,hex_icao,aircraft_type,aircraft_registration,airline_name_english,callsign_code_iata,callsign_code_icao,airline_iata,airline_icao,dep_code_iata,dep_code_icao,dep_name_english,dest_code_iata,dest_code_icao,dest_name_english,dest_lat,dest_lon,dest_altitude,arr_sched_time_utc,arr_revised_time_utc,status,dep_lat,dep_lon,dep_altitude,distance,haul,wake_turbulence_cat,actual_landing_time,landing_dist_km,actual_entry_time,entry_dist_km,post_terminal_duration_s,post_terminal_duration_min,landing_proxy_time_diff_min,landing_score,landing_candidate_count,landing_window_min,landing_source,landing_search_stage,match_confidence,landing_is_fallback,arrival_like_pre,arrival_like_post,arrival_like,arrival_pre_decrease_ratio,arrival_post_decrease_ratio,pre_points,post_points,holding_detected,holding_dist_std,holding_n_reversals,entry_source,entry_segment_source,is_matched,is_plausible_label,match_quality_flag
0,2025-07-31,Thursday,21,728678,Airbus A320,YI-ASX,I A W,IA 223,IAW223,IA,IAW,BGW,ORBI,Baghdad,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:00:00+00:00,2025-07-31 18:00:00+00:00,Arrived,33.262501,44.234600,114.0,1630.418823,medium-haul,M,2025-07-31 17:43:43+00:00,10.675306,2025-07-31 17:29:23+00:00,63.968685,860.0,14.333333,-16.283333,13.117806,3.0,90.0,distance_time_local_min_arrival_like,stage_1_pm90m,high,False,True,True,True,0.647059,NaN,52.0,1.0,True,13.936035,2,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf
1,2025-07-31,Thursday,21,7114EF,Airbus A321,UNKNOWN,Saudi Arabian,SV 261,SVA261,SV,SVA,JED,OEJN,Jeddah,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:10:00+00:00,2025-07-31 18:10:00+00:00,Arrived,21.680241,39.157436,48.0,2387.484375,medium-haul,M,2025-07-31 17:51:36+00:00,7.054359,2025-07-31 17:41:16+00:00,47.864044,620.0,10.333333,-18.400000,9.814359,1.0,90.0,distance_time_local_min_arrival_like,stage_1_pm90m,high,False,True,True,True,1.000000,NaN,44.0,1.0,False,14.526094,0,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf
2,2025-07-31,Thursday,21,040184,Boeing 737,ET-AXG,Ethiopian,ET 722,ETH722,ET,ETH,ADD,HAAB,Addis Ababa,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:10:00+00:00,2025-07-31 18:20:00+00:00,Arrived,8.977890,38.799301,7630.0,3724.830566,medium-haul,M,2025-07-31 18:23:17+00:00,12.865262,2025-07-31 18:14:19+00:00,44.443016,538.0,8.966667,3.283333,13.357762,2.0,90.0,distance_time_local_min_arrival_like,stage_1_pm90m,high,False,True,True,True,0.688889,NaN,46.0,1.0,True,13.835181,2,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf
4,2025-07-31,Thursday,21,4BA91A,Airbus A321,TC-JHZ,Turkish,TK 1098,THY9GS,TK,THY,TIV,LYTV,Tivat,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:20:00+00:00,2025-07-31 18:20:00+00:00,Arrived,42.404701,18.723301,20.0,839.713257,short-haul,M,2025-07-31 18:16:03+00:00,12.850230,2025-07-31 18:11:34+00:00,34.330212,269.0,4.483333,-3.950000,13.442730,1.0,90.0,distance_time_local_min_arrival_like,stage_1_pm90m,high,False,True,True,True,1.000000,NaN,26.0,1.0,False,5.331014,0,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf
5,2025-07-31,Thursday,21,4691C2,Airbus A320,SX-DNB,Aegean,A3 434,AEE434,A3,AEE,HER,LGIR,Heraklion,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:30:00+00:00,2025-07-31 18:40:00+00:00,Arrived,35.339699,25.180300,115.0,729.683655,short-haul,M,2025-07-31 18:25:19+00:00,11.428142,2025-07-31 18:21:48+00:00,30.468594,211.0,3.516667,-14.683333,13.630642,1.0,90.0,distance_time_local_min_arrival_like,stage_1_pm90m,high,False,True,True,True,1.000000,NaN,64.0,1.0,False,5.460646,0,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf


In [49]:
df_labeled.isna().sum()

date                               0
day_of_week                        0
hour_ist                           0
hex_icao                           0
aircraft_type                      0
aircraft_registration              0
airline_name_english               0
callsign_code_iata                 0
callsign_code_icao                 0
airline_iata                       0
airline_icao                       0
dep_code_iata                      0
dep_code_icao                      0
dep_name_english                   0
dest_code_iata                     0
dest_code_icao                     0
dest_name_english                  0
dest_lat                           0
dest_lon                           0
dest_altitude                      0
arr_sched_time_utc                 0
arr_revised_time_utc              24
status                             0
dep_lat                            0
dep_lon                            0
dep_altitude                       0
distance                           0
h

In [50]:
df_labeled = df_labeled[["hex_icao", 
                        "airline_name_english",
                         "callsign_code_iata",
                         "callsign_code_icao",
                         "haul",
                         "dep_code_iata",
                         "dep_code_icao",
                         "dep_name_english",
                         "dep_lat",
                         "dep_lon",
                         "dep_altitude",
                         "dest_code_iata",
                         "dest_code_icao",
                         "dest_name_english",
                         "dest_lat",
                         "dest_lon",
                         "dest_altitude",
                         "distance",
                         "actual_entry_time",
                         "arr_sched_time_utc",
                         "date",
                         "day_of_week",
                         "aircraft_type",
                         "aircraft_registration",
                         "wake_turbulence_cat",
                         "post_terminal_duration_min"
                         ]]

In [51]:
import pandas as pd

df_labeled['date'] = pd.to_datetime(df_labeled['date'])

monthly_counts = df_labeled.resample('MS', on='date').size().reset_index(name='flight_count')

print(monthly_counts)

         date  flight_count
0  2025-03-01          3018
1  2025-04-01             1
2  2025-05-01           945
3  2025-06-01          5023
4  2025-07-01          3791
5  2025-08-01          6033
6  2025-09-01          3517
7  2025-10-01          1174
8  2025-11-01          1384
9  2025-12-01          4045
10 2026-01-01          1952


In [52]:
df_labeled.columns

Index(['hex_icao', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'haul', 'dep_code_iata', 'dep_code_icao',
       'dep_name_english', 'dep_lat', 'dep_lon', 'dep_altitude',
       'dest_code_iata', 'dest_code_icao', 'dest_name_english', 'dest_lat',
       'dest_lon', 'dest_altitude', 'distance', 'actual_entry_time',
       'arr_sched_time_utc', 'date', 'day_of_week', 'aircraft_type',
       'aircraft_registration', 'wake_turbulence_cat',
       'post_terminal_duration_min'],
      dtype='str')